# cAPTure: selection-feature materialization

This notebook fits one inner-training-only packet preprocessor per fold and rematerializes only the five fold-training scenario sequences. Topology, labels, source rows, and node mappings must match the reviewed complete-fold graph materialization exactly.

It performs no model optimization, does not fit or materialize selection features from outer-validation scenarios, and does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`. The previously reviewed complete-fold graph collection is staged only as the alignment reference.

## 1. Mount Drive and prepare the environment

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_stage2_inputs_v2")
LOCAL_WORK_ROOT = Path("/content/capture_selection_materialization_work")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display
from utils.capture_graph_dataset import stage_capture_graph_materialization
from utils.capture_graph_selection_materialization import (
    load_completed_selection_materialization,
    run_capture_graph_selection_materialization,
    save_selection_materialization_decision,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("Selection-materialization environment is ready.")

## 2. Bind the approved immutable inputs

In [ ]:
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
GRAPH_MATERIALIZATION_RUN_ID = (
    "20260928T175821_521784Z_graph_materialization_full_dev"
)
GRAPH_INPUT_AUDIT_RUN_ID = "20260928T193308_017682Z_graph_input_full"
TRAINING_PREFLIGHT_RUN_ID = (
    "20260929T131750_353987Z_graph_training_preflight_v2"
)

PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
GRAPH_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / GRAPH_MATERIALIZATION_RUN_ID
)
GRAPH_INPUT_AUDIT_DIR = (
    DRIVE_ROOT / "graph_input_audit_runs" / GRAPH_INPUT_AUDIT_RUN_ID
)
TRAINING_PREFLIGHT_DIR = (
    DRIVE_ROOT / "graph_training_preflight_v2_runs" / TRAINING_PREFLIGHT_RUN_ID
)
MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
PREPROCESSING_SCHEMA_PATH = (
    PROJECT_ROOT / "configs/capture_preprocessing_v1.yaml"
)
GRAPH_INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)
PREFLIGHT_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_training_preflight_v2.yaml"
)
SELECTION_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_selection_materialization_v1.yaml"
)

for path in (
    PREPARED_RUN_DIR, GRAPH_MATERIALIZATION_DIR, GRAPH_INPUT_AUDIT_DIR,
    TRAINING_PREFLIGHT_DIR, MANIFEST_PATH, PACKET_SCHEMA_PATH,
    PREPROCESSING_SCHEMA_PATH, GRAPH_INPUT_CONTRACT_PATH,
    PREFLIGHT_CONFIG_PATH, SELECTION_CONFIG_PATH,
):
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")

print("Approved training preflight:", TRAINING_PREFLIGHT_DIR)
print("Selection-feature contract:", SELECTION_CONFIG_PATH)

## 3. Stage and verify the complete-fold graph reference

In [ ]:
collection = stage_capture_graph_materialization(
    source_root=GRAPH_MATERIALIZATION_DIR,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=GRAPH_INPUT_CONTRACT_PATH,
    expected_run_id=GRAPH_MATERIALIZATION_RUN_ID,
)
print("Verified graph reference:", collection.root)
print("All graph checksums verified:", collection.artifact_checksums_verified)

## 4. Run or resume selection-feature materialization

This may take several minutes. It fits only on inner-training prefixes, but rematerializes each complete fold-training scenario so topology and packet alignment can be checked over the entire sequence.

In [ ]:
SELECTION_RUN_ID = None  # Paste an existing completed ID to reuse it.
if SELECTION_RUN_ID is None:
    SELECTION_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_selection_materialization"
    )
SELECTION_OUTPUT_DIR = (
    DRIVE_ROOT / "graph_selection_materialization_runs" / SELECTION_RUN_ID
)
print("Selection-materialization run ID:", SELECTION_RUN_ID)
print("Selection-materialization output:", SELECTION_OUTPUT_DIR)

if (SELECTION_OUTPUT_DIR / "run_status.json").is_file():
    selection_report = load_completed_selection_materialization(
        SELECTION_OUTPUT_DIR,
        collection=collection,
        config_path=SELECTION_CONFIG_PATH,
    )
    print("Reused the completed immutable selection materialization.")
else:
    selection_report = run_capture_graph_selection_materialization(
        collection=collection,
        manifest_path=MANIFEST_PATH,
        packet_schema_path=PACKET_SCHEMA_PATH,
        preprocessing_schema_path=PREPROCESSING_SCHEMA_PATH,
        selection_config_path=SELECTION_CONFIG_PATH,
        preflight_config_path=PREFLIGHT_CONFIG_PATH,
        prepared_run_dir=PREPARED_RUN_DIR,
        graph_input_audit_dir=GRAPH_INPUT_AUDIT_DIR,
        preflight_dir=TRAINING_PREFLIGHT_DIR,
        output_dir=SELECTION_OUTPUT_DIR,
        local_work_root=LOCAL_WORK_ROOT,
    )
    print("Saved the immutable selection materialization.")

## 5. Review preprocessing and graph invariants

In [ ]:
rows = []
for fold, fold_report in selection_report["folds"].items():
    for scenario in fold_report["training_scenarios"]:
        scenario_report = json.loads((
            SELECTION_OUTPUT_DIR
            / fold_report["scenario_reports"][scenario]
        ).read_text(encoding="utf-8"))
        rows.append({
            "fold": fold,
            "scenario": scenario,
            "preprocessor_training_rows": fold_report["preprocessor_training_rows"],
            "active_features": fold_report["active_feature_count"],
            "masked_features": fold_report["masked_feature_count"],
            "graphs": scenario_report["graphs"],
            "edges": scenario_report["edges"],
            "shards": scenario_report["shard_count"],
            "compressed_GiB": scenario_report["compressed_shard_bytes"] / 1024**3,
            "wall_seconds": scenario_report["wall_seconds"],
            "peak_RSS_GiB": (
                scenario_report["peak_sampled_rss_bytes"] / 1024**3
                if scenario_report["peak_sampled_rss_bytes"] is not None
                else None
            ),
            **fold_report["invariants"][scenario],
        })
display(pd.DataFrame(rows))
print("Status:", selection_report["status"])
print("Materialized edges:", selection_report["total_materialized_edges"])
print("Expected unique development packets:",
      selection_report["expected_unique_development_packets"])
print("Compressed shards (GiB):",
      selection_report["compressed_shard_bytes"] / 1024**3)
print("Selection preprocessing fitted:",
      selection_report["selection_preprocessing_fitted"])
print("Selection features materialized:",
      selection_report["selection_features_materialized"])
print("Model training performed:",
      selection_report["model_training_performed"])
print("Outer-validation scenarios materialized:",
      selection_report["outer_validation_scenarios_materialized"])
print("Outer-validation scenarios used for fit:",
      selection_report["outer_validation_scenarios_used_for_fit"])
print("Complete-fold reference checksums verified:",
      selection_report["complete_fold_reference_checksums_verified"])
print("Held-out scenarios accessed:",
      selection_report["held_out_scenarios_accessed"])

## 6. Record the manual materialization decision

Approval permits binding the training runner to these tensors. It still does not authorize model optimization.

In [ ]:
APPROVE_SELECTION_MATERIALIZATION = False
SELECTION_REVIEW_NOTE = "Unapproved"
DECISION_PATH = SELECTION_OUTPUT_DIR / "selection_materialization_decision.json"

if DECISION_PATH.exists():
    decision = json.loads(DECISION_PATH.read_text(encoding="utf-8"))
    print("Existing selection decision:", json.dumps(decision, indent=2))
else:
    if not APPROVE_SELECTION_MATERIALIZATION:
        raise RuntimeError(
            "Review the materialization before explicitly approving it."
        )
    decision = save_selection_materialization_decision(
        SELECTION_OUTPUT_DIR,
        approved=True,
        review_note=SELECTION_REVIEW_NOTE,
    )
    print("Saved selection decision:", json.dumps(decision, indent=2))